In [ ]:
%pip install peft transformers datasets accelerate gradio

In [ ]:
%pip install -U transformers

In [ ]:
%pip install -U peft accelerate


In [ ]:
import transformers
print(transformers.__version__)
print(transformers.__file__)


In [ ]:
import transformers
print(transformers.__version__)

In [14]:
%pip uninstall -y transformers peft accelerate
%pip install -U transformers peft accelerate


Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
Found existing installation: peft 0.15.2
Uninstalling peft-0.15.2:
  Successfully uninstalled peft-0.15.2
Found existing installation: accelerate 1.7.0
Uninstalling accelerate-1.7.0:
  Successfully uninstalled accelerate-1.7.0
Note: you may need to restart the kernel to use updated packages.
  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached peft-0.15.2-py3-none-any.whl.metadata (13 kB)
  Using cached accelerate-1.7.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-4.51.3-py3-none-any.whl (10.4 MB)
Using cached peft-0.15.2-py3-none-any.whl (411 kB)
Using cached accelerate-1.7.0-py3-none-any.whl (362 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Tema del Dataset: Curiosidades Científicas de la Biología de las Plantas

In [17]:
import json
from datasets import Dataset
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType

# 1. Cargar dataset JSON (asegúrate de tener el archivo 'biologia_plantas.json' en el directorio)
with open("biologia_plantas.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

# 2. Cargar tokenizer y modelo base
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 3. Añadir token padding si falta
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<pad>'})
    model.resize_token_embeddings(len(tokenizer))

# 4. Añadir tokens especiales
special_tokens = ['<|user|>', '<|assistant|>']
tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})
model.resize_token_embeddings(len(tokenizer))

# 5. Construcción de prompts para fine-tuning
def build_prompt(example):
    return {"text": f"<|user|>\n{example['instruction']}\n<|assistant|>\n{example['response']}"}

dataset = dataset.map(build_prompt, remove_columns=["instruction", "response"])

# 6. Tokenización del dataset
def tokenize(example):
    tokenized = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

dataset = dataset.map(tokenize, batched=True)

# 7. Configuración LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

# 8. Preparar argumentos de entrenamiento con import completo para evitar conflictos
training_args = transformers.TrainingArguments(
    output_dir="./results_bio",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    evaluation_strategy="epoch",  # evaluamos cada epoch
    save_strategy="epoch",
    logging_dir="./logs",
    report_to="none"
)

# 9. Data collator para causal LM
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 10. Crear Trainer y entrenar
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

# 11. Guardar modelo y tokenizer fine-tuneados
trainer.save_model("./results_bio")
tokenizer.save_pretrained("./results_bio")


Map:   0%|          | 0/101 [00:00<?, ? examples/s]

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

🖼️ Interfaz Gradio para Inferencia

In [ ]:
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model = AutoModelForCausalLM.from_pretrained("./results_bio")
tokenizer = AutoTokenizer.from_pretrained("./results_bio")

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def responder(pregunta):
    prompt = f"<|user|>\n{pregunta}\n<|assistant|>\n"
    respuesta = pipe(prompt, max_new_tokens=100, do_sample=True, temperature=0.7)[0]["generated_text"]
    return respuesta.split("<|assistant|>\n")[-1]

gr.Interface(fn=responder, inputs="text", outputs="text", title="Asistente de Biología de Plantas").launch()
